---
description: Add tamper-evident Proof-of-Time sealing to LLM traces sent to Langfuse via OpenTelemetry, enabling AI Act Art.12 / DORA-compatible integrity verification.
category: Integrations
---

# Tamper-Evident LLM Tracing with ttt-otel and Langfuse

This cookbook shows how to add **Proof-of-Time (PoT) sealing** to an OpenTelemetry pipeline that exports LLM traces to Langfuse.  
Each span is hashed with blake3 and sealed into a tamper-evident chain via [api.kenosian.com](https://api.kenosian.com).  
Raw prompt/completion text never leaves your process — only the hash is sent to the sealing server.

**Why this matters**

- **EU AI Act Art.12** requires high-risk AI systems to keep tamper-evident logs of inputs/outputs.  
- **DORA** (Digital Operational Resilience Act) mandates immutable audit trails for financial-sector AI.  
- The PoT chain lets you prove, after the fact, that a trace was not modified — without storing raw content anywhere outside your own environment.

**Architecture**

```
Your app
  └── OTel TracerProvider
        ├── BatchSpanProcessor → OTLPSpanExporter → Langfuse  (trace visibility)
        └── TTTSpanProcessor   → api.kenosian.com /pot/generate  (integrity seal)
```

Both processors run in parallel — Langfuse receives the full span, the sealing server receives only the blake3 hash.

## Step 1: Install Dependencies

In [ ]:
%pip install \
    opentelemetry-sdk \
    opentelemetry-exporter-otlp-proto-http \
    blake3 \
    httpx \
    "git+https://github.com/Helm-Protocol/ttt-otel"

## Step 2: Configure Environment Variables

Set your Langfuse project keys and your Proof-of-Time API key.  
Get your Langfuse keys from [cloud.langfuse.com](https://cloud.langfuse.com) → Project Settings.  
Get your PoT API key from [kenosian.com](https://kenosian.com).

In [ ]:
import os
import base64

# Langfuse project keys
os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-<YOUR_LANGFUSE_PUBLIC_KEY>"
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-<YOUR_LANGFUSE_SECRET_KEY>"
os.environ["LANGFUSE_HOST"]       = "https://cloud.langfuse.com"  # EU region
# Other regions:
#   US:    https://us.cloud.langfuse.com
#   Japan: https://jp.cloud.langfuse.com
#   HIPAA: https://hipaa.cloud.langfuse.com

# Proof-of-Time API key (kenosian.com)
os.environ["TTT_API_KEY"] = "<YOUR_TTT_API_KEY>"

## Step 3: Build the TracerProvider

Add both processors to a single `TracerProvider`:

1. **`BatchSpanProcessor` + `OTLPSpanExporter`** — ships the full span to Langfuse over OTLP/HTTP.
2. **`TTTSpanProcessor`** — seals each span's content-hash into the PoT chain asynchronously.  
   The processor runs two background worker threads and never blocks `on_end()`.

In [ ]:
import base64
import os

from opentelemetry import trace
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter

from ttt_otel import TTTSpanProcessor

LANGFUSE_PUBLIC_KEY = os.environ["LANGFUSE_PUBLIC_KEY"]
LANGFUSE_SECRET_KEY = os.environ["LANGFUSE_SECRET_KEY"]
LANGFUSE_HOST       = os.environ["LANGFUSE_HOST"]
TTT_API_KEY         = os.environ["TTT_API_KEY"]

# Build Langfuse Basic Auth header
langfuse_auth = base64.b64encode(
    f"{LANGFUSE_PUBLIC_KEY}:{LANGFUSE_SECRET_KEY}".encode()
).decode()

# --- TracerProvider ----------------------------------------------------------
resource = Resource.create({"service.name": "my-llm-app"})
provider = TracerProvider(resource=resource)

# 1) Langfuse OTLP exporter
otlp_exporter = OTLPSpanExporter(
    endpoint=f"{LANGFUSE_HOST}/api/public/otel/v1/traces",
    headers={"Authorization": f"Basic {langfuse_auth}"},
)
provider.add_span_processor(BatchSpanProcessor(otlp_exporter))

# 2) Proof-of-Time sealing processor
ttt_processor = TTTSpanProcessor(
    api_key=TTT_API_KEY,
    server_url="https://api.kenosian.com",
    # Optional: persist seals to disk for durable audit trail
    # persist_path="/var/log/pot_seals.jsonl",
)
provider.add_span_processor(ttt_processor)

trace.set_tracer_provider(provider)

print("TracerProvider configured — Langfuse OTLP + PoT sealing active.")

## Step 4: Instrument an LLM Agent Trace

Create spans as you normally would with OpenTelemetry.  
Use the [GenAI semantic conventions](https://opentelemetry.io/docs/specs/semconv/gen-ai/) for best Langfuse display: `gen_ai.system`, `gen_ai.prompt`, `gen_ai.completion`.

The `TTTSpanProcessor` extracts these attributes, hashes them locally, and seals the hash.  
No prompt or completion text is transmitted to the sealing server.

In [ ]:
import time

tracer = trace.get_tracer("my-llm-app")

# Simulate a simple two-step agent: plan → tool call
with tracer.start_as_current_span("agent.session"):

    with tracer.start_as_current_span("agent.plan") as plan_span:
        plan_span.set_attribute("gen_ai.system", "gpt-4o")
        plan_span.set_attribute("gen_ai.prompt", "Summarise Q1 sales data.")
        plan_span.set_attribute("gen_ai.completion",
                                "I will query the database and produce a summary.")
        time.sleep(0.05)  # simulate model latency

    with tracer.start_as_current_span("agent.tool_call") as tool_span:
        tool_span.set_attribute("gen_ai.system", "gpt-4o")
        tool_span.set_attribute("gen_ai.prompt",
                                "SELECT SUM(revenue) FROM sales WHERE quarter='Q1'")
        tool_span.set_attribute("gen_ai.completion", "Total: $4,821,000")
        time.sleep(0.05)

print("Spans emitted — flushing to Langfuse and sealing into PoT chain...")

# Wait for background PoT sealing to complete before inspecting seals
ttt_processor.force_flush(timeout_millis=15_000)
print("Done.")

## Step 5: View Traces in Langfuse

Open your [Langfuse dashboard](https://cloud.langfuse.com) and navigate to **Traces**.  
You should see the `agent.session` trace with its two child spans.

Each span also has a corresponding seal record stored in `ttt_processor` (and optionally on disk via `persist_path`).  
The seal contains the `content_hash`, `pot_hash`, `timestamp_ns`, and `prev_content_hash` — forming a chain you can verify independently of Langfuse.

## Step 6: Inspect Seal Records

In [ ]:
import json

seals = ttt_processor.all_seals()
print(f"Sealed {len(seals)} span(s):\n")

for content_hash_hex, record in seals.items():
    resp = record["pot_response"]
    prev = record.get("prev_content_hash") or "(none — chain start)"
    print(
        f"  span          : {record['fields'].get('name', '?')}\n"
        f"  content_hash  : {content_hash_hex[:32]}...\n"
        f"  pot_hash      : {resp.get('pot_hash', '?')[:32]}...\n"
        f"  timestamp_ns  : {resp.get('timestamp_ns', '?')}\n"
        f"  time_mode     : {resp.get('current_mode', resp.get('mode', '?'))}\n"
        f"  prev          : {str(prev)[:32]}...\n"
    )

## Step 7: Verify a Seal (Tamper Detection)

Use `verify()` to confirm that a span's canonical fields match what was sealed on the server.  
If any field was modified after sealing, the content-hash will differ and `valid` will be `False`.

The example below shows:
1. Verifying the original span — expected result: `valid=True`.
2. Verifying a tampered version of the same span — expected result: `valid=False`.

In [ ]:
from ttt_otel import verify
from ttt_otel._hash import content_hash

# --- Original fields (as sealed) -------------------------------------------
original_fields = {
    "trace_id":             "0000000000000000000000000cafebabe",
    "span_id":              "0000000000000001",
    "name":                 "agent.plan",
    "model":                "gpt-4o",
    "input_hash":           content_hash({"v": "Summarise Q1 sales data."}),
    "output_hash":          content_hash({"v": "I will query the database and produce a summary."}),
    "start_time_unix_nano": 1_700_000_000_000_000_000,
    "end_time_unix_nano":   1_700_000_000_050_000_000,
    "parent_span_id":       "",
}

result_original = verify(
    original_fields,
    api_key=TTT_API_KEY,
)
print("Original span  →", "VERIFIED" if result_original["valid"] else "NOT FOUND",
      f"| chain_intact={result_original['chain_intact']}")

# --- Tampered fields (output changed) ----------------------------------------
tampered_fields = dict(original_fields)
tampered_fields["output_hash"] = content_hash({"v": "I will skip the query and return a fake number."})  # forged!

result_tampered = verify(
    tampered_fields,
    api_key=TTT_API_KEY,
)
print("Tampered span  →", "VERIFIED" if result_tampered["valid"] else "TAMPER DETECTED",
      f"| found={result_tampered['found']}")

## How the Privacy Model Works

| What is sent to Langfuse | What is sent to the PoT server |
|--------------------------|--------------------------------|
| Full span (name, attributes, timing, trace/span IDs) | Only the blake3 content-hash of normalised fields |
| Stored in your Langfuse project | Stored as an event in the PoT chain (no raw text) |

The canonical fields hashed by `TTTSpanProcessor` are:

- `trace_id`, `span_id`, `name`, `model`
- `input_hash` — blake3 of the raw prompt (computed locally, not transmitted)
- `output_hash` — blake3 of the raw completion (computed locally, not transmitted)
- `start_time_unix_nano`, `end_time_unix_nano`, `parent_span_id`

This means the sealing server cannot reconstruct your prompts even if compromised.

## AI Act Art.12 / DORA Context

**EU AI Act Art.12** (in force from August 2026 for high-risk systems) requires that high-risk AI systems automatically record events with sufficient detail to identify causes of malfunctions and enable post-market monitoring.  
A tamper-evident seal on every inference span provides an independently verifiable audit record.

**DORA Art.9** requires that financial entities maintain ICT-related incident logs that are protected against modification and unauthorised access.  
The PoT chain gives each log entry a cryptographic timestamp and a causal link to the previous entry, making undetected modification computationally infeasible.

## Further Resources

- [ttt-otel repository](https://github.com/Helm-Protocol/ttt-otel)
- [Langfuse OpenTelemetry integration docs](https://langfuse.com/integrations/native/opentelemetry)
- [GenAI semantic conventions](https://opentelemetry.io/docs/specs/semconv/gen-ai/)
- [EU AI Act Art.12 text](https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:32024R1689)